In [1]:
import boto3
import pickle
from dotenv import load_dotenv
import os
load_dotenv()

def load(bucket_name, model_path):
    s3_client = boto3.client(
        's3',
        aws_access_key_id=os.getenv('NOTEBOOK_ACCESS_KEY'),
        aws_secret_access_key=os.getenv('NOTEBOOK_ACCESS_KEY_SECRET')
    )   
    response = s3_client.get_object(Bucket=bucket_name, Key=model_path)
    model = pickle.loads(response['Body'].read())
    return model


Modelo cargado

In [2]:
bucket_name="chicago-inspections-analytics"
model_path="selected-model/select_model.pkl"
test_dataset_path="dataset/test/test_dataset.pkl"
test_target_path="dataset/test/test_target.pkl"
train_dataset_path='dataset/train/train_dataset.pkl'
train_target_path='dataset/train/train_target.pkl'
model=load(bucket_name, model_path)# Carga el modelo desde S3.
test_dataset=load(bucket_name,test_dataset_path)
test_target=load(bucket_name,test_target_path)

In [3]:
test_target.head()

151258    fail
151259    pass
151260    pass
151261    fail
151262    pass
Name: remainder__results, dtype: object

In [4]:
# Convertir test_target (que es una Serie) a un DataFrame
test_target_df = test_target.to_frame()



# Ver las primeras filas del DataFrame
test_target_df.head()


,remainder__results
151258,fail
151259,pass
151260,pass
151261,fail
151262,pass


In [5]:
test_dataset.head(10)




,facility_type__facility_type_BANQUET HALL,facility_type__facility_type_Bakery,facility_type__facility_type_Catering,facility_type__facility_type_Children's Services Facility,facility_type__facility_type_GAS STATION,facility_type__facility_type_Golden Diner,facility_type__facility_type_Grocery Store,facility_type__facility_type_Hospital,facility_type__facility_type_Liquor,facility_type__facility_type_Long Term Care,...,risk__risk_medium,remainder__latitude,remainder__longitude,remainder__month,remainder__year,remainder__day_of_month,remainder__week_of_year,remainder__week_day,remainder__weekend,remainder__day_of_week
151258,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,41.9591,-87.682392,6,2018,14,24,1,0,3
151259,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,41.886377,-87.624382,6,2018,14,24,1,0,3
151260,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,41.768926,-87.644656,6,2018,14,24,1,0,3
151261,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,41.796235,-87.630405,6,2018,14,24,1,0,3
151262,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,41.923473,-87.639349,6,2018,14,24,1,0,3
151263,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,41.878223,-87.634247,6,2018,14,24,1,0,3
151264,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,41.852397,-87.668377,6,2018,14,24,1,0,3
151265,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,41.880357,-87.662082,6,2018,14,24,1,0,3
151266,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,41.893063,-87.628265,6,2018,14,24,1,0,3
151267,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,41.822548,-87.616816,6,2018,14,24,1,0,3


In [6]:
predicciones = model.predict(test_dataset)


In [7]:
# Mapear las predicciones de 0 y 1 a 'fail' y 'pass'
predicciones_clasificadas = ['fail' if pred == 0 else 'pass' for pred in predicciones]


In [8]:
# Agregar las predicciones al DataFrame
test_dataset['predicciones'] = predicciones_clasificadas


In [9]:
# Contar las ocurrencias de "pass" y "fail"
conteo_predicciones = test_dataset['predicciones'].value_counts()

# Imprimir el conteo
print(conteo_predicciones)


predicciones
pass    64826
Name: count, dtype: int64


In [10]:
test_target_df.value_counts()

remainder__results
pass                  48496
fail                  16330
Name: count, dtype: int64

SVM

In [11]:
import pandas as pd
import numpy as np
import pickle
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score


# 📊 Función para entrenar y evaluar SVM
def train_evaluate_svm(X_train, X_test, y_train, y_test, normalize=False):
    if normalize:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    # Establecer n_jobs=-1 para utilizar todos los hilos disponibles
    svm_model = SVC(kernel="rbf", probability=True, random_state=42)
    svm_model.fit(X_train, y_train)

    y_pred = svm_model.predict(X_test)
    y_prob = svm_model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    return accuracy, auc

# 📌 Evaluación sin normalización
X_train = load(bucket_name, train_dataset_path)  # Carga el dataset desde S3.
y_train = load(bucket_name, train_target_path)  # Carga el target desde S3.
X_test = load(bucket_name, test_dataset_path)  # Carga el dataset de prueba desde S3.
y_test = load(bucket_name, test_target_path)  # Carga el target de prueba desde S3.

accuracy_no_norm, auc_no_norm = train_evaluate_svm(X_train, X_test, y_train, y_test, normalize=False)

# 📌 Evaluación con normalización
accuracy_norm, auc_norm = train_evaluate_svm(X_train, X_test, y_train, y_test, normalize=True)

# 📊 Comparativa de resultados
print("🔹 Sin Normalización:")
print(f"   - Precisión: {accuracy_no_norm:.4f}")
print(f"   - AUC: {auc_no_norm:.4f}")

print("\n🔹 Con Normalización:")
print(f"   - Precisión: {accuracy_norm:.4f}")
print(f"   - AUC: {auc_norm:.4f}")


🔹 Sin Normalización:
   - Precisión: 0.7481
   - AUC: 0.5057

🔹 Con Normalización:
   - Precisión: 0.7483
   - AUC: 0.5157


In [12]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# 📊 Función para entrenar y evaluar la red neuronal
def train_evaluate_nn(X_train, X_test, y_train, y_test, normalize=False):
    if normalize:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    # Definir el modelo de red neuronal
    model = Sequential()
    model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))  # Capa de entrada
    model.add(Dense(32, activation='relu'))  # Capa oculta
    model.add(Dense(1, activation='sigmoid'))  # Capa de salida para clasificación binaria

    # Compilar el modelo
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

    # Entrenar el modelo
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    # Evaluar el modelo
    y_pred = (model.predict(X_test) > 0.5).astype("int32")  # Predecir con un umbral de 0.5
    y_prob = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    return accuracy, auc

# 📌 Evaluación sin normalización
X_train = load(bucket_name, train_dataset_path)  # Carga el dataset desde S3.
y_train = load(bucket_name, train_target_path)  # Carga el target desde S3.
X_test = load(bucket_name, test_dataset_path)  # Carga el dataset de prueba desde S3.
y_test = load(bucket_name, test_target_path)  # Carga el target de prueba desde S3.

accuracy_no_norm, auc_no_norm = train_evaluate_nn(X_train, X_test, y_train, y_test, normalize=False)

# 📌 Evaluación con normalización
accuracy_norm, auc_norm = train_evaluate_nn(X_train, X_test, y_train, y_test, normalize=True)

# 📊 Comparativa de resultados
print("🔹 Sin Normalización:")
print(f"   - Precisión: {accuracy_no_norm:.4f}")
print(f"   - AUC: {auc_no_norm:.4f}")

print("\n🔹 Con Normalización:")
print(f"   - Precisión: {accuracy_norm:.4f}")
print(f"   - AUC: {auc_norm:.4f}")


2025-02-08 01:18:15.613007: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-08 01:18:15.621570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738999095.628958   19535 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738999095.631167   19535 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-08 01:18:15.639758: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

ValueError: Invalid dtype: object

In [ ]:
import tensorflow as tf
print("GPU Available:", tf.test.is_gpu_available())


2025-02-07 18:10:56.130291: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-07 18:10:56.143342: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738973456.154277    9341 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738973456.157804    9341 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-07 18:10:56.169479: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

ImportError: cannot import name 'builder' from 'google.protobuf.internal' (/usr/lib/python3/dist-packages/google/protobuf/internal/__init__.py)